I- Filter Eqasim outputs (CSV + optional GPKG)

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Set

import pandas as pd
import geopandas as gpd

# ============================================================
# Filter Eqasim outputs (CSV + optional GPKG)
# ============================================================

# ------------------------------
# CONFIG
# ------------------------------
BASE_PATH = Path(".")  # Folder containing Eqasim outputs (17*.csv, 17*.gpkg)
OUTPUT_PATH = Path("./eqasim_output_filtered")  # Output folder

# Step 1: the 8 peri-urban communes (INSEE codes)
PERIURBAN_COMMUNES_8: set[int] = {
    17059, 17245, 17109, 17194, 17373, 17315, 17136, 17420
}

# Step 2: the 28 communes of CdA La Rochelle (INSEE codes) (BANATIC)
CDA_COMMUNES_28: set[int] = {
    17010, 17028, 17059, 17094, 17109, 17136, 17142, 17153,
    17190, 17193, 17194, 17200, 17222, 17245, 17264, 17274,
    17291, 17300, 17315, 17373, 17391, 17407, 17413, 17414,
    17420, 17443, 17466, 17483
}

# ------------------------------
# HELPERS
# ------------------------------
def ensure_dir(path: Path) -> None:
    """Create a directory if it does not exist."""
    path.mkdir(parents=True, exist_ok=True)


def read_csv_file(filename: str) -> pd.DataFrame:
    """Read a ';' separated CSV from BASE_PATH."""
    return pd.read_csv(BASE_PATH / filename, sep=";")


def try_filter_gpkg(
    in_name: str,
    out_name: str,
    key_col: str,
    keep_ids: Set[str],
) -> None:
    """Filter a GeoPackage by key column membership and write a new GPKG."""
    in_path = BASE_PATH / in_name
    if not in_path.exists():
        print(f"[WARN] {in_name} not found -> skip")
        return

    gdf = gpd.read_file(in_path)
    if key_col not in gdf.columns:
        print(f"[WARN] {in_name} does not contain column '{key_col}' -> skip")
        return

    gdf[key_col] = gdf[key_col].astype(str)
    out = gdf[gdf[key_col].isin(keep_ids)]
    out.to_file(OUTPUT_PATH / out_name, driver="GPKG")
    print(f"[OK] Wrote {out_name} ({len(out)} rows)")


def filter_and_write_csv_if_exists(
    filename: str,
    out_filename: str,
    id_col: str,
    selected_ids: Set[str],
) -> None:
    """Filter a CSV by id_col membership if the file exists."""
    in_path = BASE_PATH / filename
    if not in_path.exists():
        print(f"[WARN] {filename} not found -> skip")
        return

    df = read_csv_file(filename)
    df[id_col] = df[id_col].astype(str)
    out = df[df[id_col].isin(selected_ids)]
    out.to_csv(OUTPUT_PATH / out_filename, sep=";", index=False)
    print(f"[OK] Wrote {out_filename} ({len(out)} rows)")


# ------------------------------
# RUN
# ------------------------------
ensure_dir(OUTPUT_PATH)

households = read_csv_file("17households.csv")
persons = read_csv_file("17persons.csv")
activities = read_csv_file("17activities.csv")

# Normalise identifiers (avoid float/int surprises)
households["household_id"] = households["household_id"].astype(str)
persons["person_id"] = persons["person_id"].astype(str)
persons["household_id"] = persons["household_id"].astype(str)

# commune_id may appear as float in some exports -> coerce to nullable integer
activities["person_id"] = activities["person_id"].astype(str)
activities["commune_id"] = pd.to_numeric(activities["commune_id"], errors="coerce").astype("Int64")

# STEP 1: residents of the 8 peri-urban communes
households["commune_id"] = pd.to_numeric(households["commune_id"], errors="coerce").astype("Int64")
base_household_ids: Set[str] = set(
    households.loc[households["commune_id"].isin(PERIURBAN_COMMUNES_8), "household_id"]
)
base_person_ids: Set[str] = set(
    persons.loc[persons["household_id"].isin(base_household_ids), "person_id"]
)

# STEP 2: persons whose ALL activities are within CdA (28 communes)
in_cda_row = activities["commune_id"].isin(CDA_COMMUNES_28) & activities["commune_id"].notna()
all_acts_in_cda = in_cda_row.groupby(activities["person_id"]).all()
cda_person_ids: Set[str] = set(all_acts_in_cda[all_acts_in_cda].index.astype(str))

# UNION: step 1 + step 2
selected_person_ids: Set[str] = base_person_ids | cda_person_ids

# Associated households (households of the selected persons)
selected_household_ids: Set[str] = set(
    persons.loc[persons["person_id"].isin(selected_person_ids), "household_id"]
)

print("========== SUMMARY ==========")
print(f"[INFO] Step 1 persons (8 communes): {len(base_person_ids)}")
print(f"[INFO] Step 2 persons (all activities in CdA): {len(cda_person_ids)}")
print(f"[INFO] Total selected persons (union): {len(selected_person_ids)}")
print(f"[INFO] Total selected households: {len(selected_household_ids)}")
print("=============================")

# EXPORT CSV
households_out = households[households["household_id"].isin(selected_household_ids)]
households_out.to_csv(OUTPUT_PATH / "17households_filtered.csv", sep=";", index=False)

persons_out = persons[persons["person_id"].isin(selected_person_ids)]
persons_out.to_csv(OUTPUT_PATH / "17persons_filtered.csv", sep=";", index=False)

# Activities (keep commune_id as Int64)
activities_out = activities[activities["person_id"].isin(selected_person_ids)].copy()
activities_out.to_csv(OUTPUT_PATH / "17activities_filtered.csv", sep=";", index=False)

# Optional CSV files
filter_and_write_csv_if_exists(
    filename="17trips.csv",
    out_filename="17trips_filtered.csv",
    id_col="person_id",
    selected_ids=selected_person_ids,
)
filter_and_write_csv_if_exists(
    filename="17pt_legs.csv",
    out_filename="17pt_legs_filtered.csv",
    id_col="person_id",
    selected_ids=selected_person_ids,
)

# EXPORT GPKG (optional)
try_filter_gpkg("17homes.gpkg", "17homes_filtered.gpkg", "household_id", selected_household_ids)
try_filter_gpkg("17activities.gpkg", "17activities_filtered.gpkg", "person_id", selected_person_ids)
try_filter_gpkg("17trips.gpkg", "17trips_filtered.gpkg", "person_id", selected_person_ids)
try_filter_gpkg("17commutes.gpkg", "17commutes_filtered.gpkg", "person_id", selected_person_ids)

print("[OK] Filtering completed.")

========== SUMMARY ==========
[INFO] Step 1 persons (8 communes): 1647
[INFO] Step 2 persons (all activities in CdA): 17159
[INFO] Total selected persons (union): 17602
[INFO] Total selected households: 10052
[OK] Wrote 17trips_filtered.csv (60298 rows)
[OK] Wrote 17pt_legs_filtered.csv (6981 rows)
[OK] Wrote 17homes_filtered.gpkg (10052 rows)
[OK] Wrote 17activities_filtered.gpkg (77900 rows)
[OK] Wrote 17trips_filtered.gpkg (60298 rows)
[OK] Wrote 17commutes_filtered.gpkg (6490 rows)
[OK] Filtering completed.


II- Filter MATSim population XML (keep selected persons)

In [2]:
import gzip
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

# ============================================================
# 2 — Filter MATSim population XML (keep selected persons)
# ============================================================

BASE_PATH = Path(".")
POP_XML_IN = BASE_PATH / "17population.xml.gz"

OUT_DIR = Path("./eqasim_output_filtered")  # Same as OUTPUT_PATH from Step 2
PERSONS_FILTERED_CSV = OUT_DIR / "17persons_filtered.csv"
POP_XML_OUT = OUT_DIR / "17population_filtered.xml.gz"

# Load person ids to keep
persons_df = pd.read_csv(PERSONS_FILTERED_CSV, sep=";")
keep_ids = set(persons_df["person_id"].astype(str))

# Read gzipped MATSim population XML
with gzip.open(POP_XML_IN, "rb") as f:
    tree = ET.parse(f)
root = tree.getroot()

# Remove persons not in the selected set
all_persons = list(root.findall("person"))  # list() to avoid modifying while iterating
for person in all_persons:
    if person.attrib.get("id") not in keep_ids:
        root.remove(person)

# Write filtered population back to gzipped XML
with gzip.open(POP_XML_OUT, "wb") as f_out:
    tree.write(f_out, encoding="utf-8", xml_declaration=True)

print(f"[OK] Wrote: {POP_XML_OUT}")


[OK] Wrote: eqasim_output_filtered\17population_filtered.xml.gz
